In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np

sys.path.append(str(Path().resolve().parent))
from src.data_processing.data_processing import make_data_from_faculty, clean_data, make_data_from_program, make_features, process_data
from src.data_processing.data_processing import main as main_data

In [2]:
df_clean = main_data()

/mnt/d/ml_project/ml_project_2026/src/data_processing/data_processing.py:8: DtypeWarning: Columns (0: course, 1: module, 2: academic_year) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ';')


In [7]:
df_clean = clean_data(df_clean)
df_clean = process_data(df_clean)



In [8]:
df_clean.head()

,student_id_hash,campus,faculty,program,education_level,course,group,place_type,subject_name,exam_type,grade_10,absence_status,module,academic_year,student_status
0,657b2a6069af97615b4c82205863561b,Москва,факультет экономических наук,Финансовый инжиниринг,Магистратура,2,МФИИН231,Коммерческие,Дисциплина: Финансовые рынки: проблемы и решения,Первая сдача,6,attendance,1,2024,graduated
1,394b76e0ea7739201d39f8c139808de3,Москва,Высшая школа бизнеса,Управление бизнесом,Бакалавриат,4,БМБ2102,Коммерческие,Дисциплина: Управление организационной жизнест...,Первая сдача,7,attendance,1,2024,graduated
2,a6b5e298d01156513fec7056f21313c5,Москва,факультет права,Юрист в правосудии,Магистратура,2,МЮП231,Бюджетные,Дисциплина: Отбор и назначение судей,Первая сдача,8,attendance,1,2024,graduated
3,592333b03350965546a17ae33cd4d3d8,Москва,факультет права,Юрист в бизнесе,Магистратура,2,МЮБ232,Коммерческие,Дисциплина: Экономический анализ права,Первая сдача,8,attendance,1,2024,graduated
4,871132d017f31ba4bf97371ac228c638,Москва,факультет мировой экономики и мировой политики,Экономика окружающей среды и устойчивое развитие,Магистратура,2,МЭОС231,Коммерческие места для иностранных граждан,Научно-исследовательский семинар: Семинар наст...,Первая сдача,5,attendance,1,2024,graduated


In [5]:
df_clean['exam_type'].unique()

<StringArray>
[                     'Первая сдача',                         'Пересдача',
                  'Полный перезачет',             'Пересдача с комиссией',
 'Пересдача по уважительной причине',               'Частичный перезачет']
Length: 6, dtype: str

In [6]:
import statistics

def make_features_loc(df : pd.DataFrame) -> pd.DataFrame:
    result = []
    
    for student_id_hash, group in df.groupby('student_id_hash'):
        grades = group['grade_10'].tolist()
        avg_grade = sum(grades) / len(grades)

        
        student_status = group['student_status'].iloc[0]
        
        
        result.append({
            'student__id_hash': student_id_hash,
            # 'grades_list': grades,
            'avg_grade': avg_grade,
            # 'median_grade': statistics.median(grades),
            # 'min_grade': min(grades),
            # 'max_grade': max(grades),
            'count_grades': len(grades),
            # 'count_retake': len(group[group['exam_type'] == 'Пересдача']),
            'proportion_retake': len(group[group['exam_type'] == 'Пересдача'])/len(grades),
            # 'count_retake_com': len(group[group['exam_type'] == 'Пересдача с комиссией']),
            'proportion_retake_com': len(group[group['exam_type'] == 'Пересдача с комиссией'])/len(grades),
            'program' : group['program'].iloc[0],
            'course': group['course'].iloc[0],
            'place_type': group['place_type'].iloc[0],
            'student_status': student_status

        })

    
    
    return pd.DataFrame(result)

df_encoded = make_features_loc(df_clean)

df_encoded['course'] = df_encoded['course'].astype('category')

In [7]:
df_encoded['count_grades'].unique()
# print(df_encoded.loc[0].notna().sum())

# cols = [s for s in df_encoded.columns if '3_2' in s]


# df_encoded_part = df_encoded[cols]

array([ 24, 115,  14,  22,  16,  41,  18,  52,  12,  55,  15,  13,  21,
        58,   9,  50,  19,  32,  26,  17,  61,  35,  28,  40,  25,  34,
        11,   1,  65,  20,  38,  31,  10,   4,   8,  49,  39,   7,   6,
         3,  37,  62,  56,  51,  47,  29,  64,  57,  53,  33,  68,   5,
        27,   2,  42,  45,  59,  23,  54,  44,  43,  36,  63,  66,  30,
        60,  46,  98,  48,  70,  75,  91, 100,  69,  67,  72,  76,  77,
        74,  82,  73,  92,  71,  81, 109,  85, 107,  94, 126, 102,  93,
        96,  84,  90, 157, 121,  88, 149,  78,  80,  89,  83,  87,  95,
        79, 125, 118, 132, 154, 106, 104,  86,  97, 112, 111, 152, 114,
        99, 103, 131, 120, 113, 124, 108])

In [8]:
def str_to_int_stadent_status (df : pd.DataFrame):
    df["student_status"] = df["student_status"].map({
        "study": 0,
        "graduated": 0,
        "expelled": 1,
        "leave" :  1
    })
    return df

In [9]:
df_encoded = str_to_int_stadent_status(df_encoded)


In [10]:
from scipy.stats import pearsonr

def trend_corr(grades):
    clean_grades = np.array([x for x in grades if pd.notna(x)], dtype=float)
    x = np.arange(len(clean_grades))
    if len(clean_grades) < 2 or np.std(clean_grades)==0:
        return 0
    return pearsonr(x, clean_grades)[0]

def make_dynamic_features_loc(df : pd.DataFrame) -> pd.Series:
    df_sorted = df.sort_values(['student_id_hash', 'course', 'module'])
    g = df_sorted.groupby('student_id_hash')['grade_10']
    pearson = g.apply(trend_corr)
    return pearson

df_encoded['pearson'] = df_encoded['student__id_hash'].map(make_dynamic_features_loc(df_clean))

In [11]:
df_encoded

,student__id_hash,avg_grade,count_grades,proportion_retake,proportion_retake_com,program,course,place_type,student_status,pearson
0,00000d86dad0d3c197341f776bb61a1d,8.166667,24,0.000000,0.000000,Математика,2,Бюджетные,0,-0.104406
1,0001dd443aff5c0800694730323c3d7d,6.973913,115,0.034783,0.008696,Реклама и связи с общественностью,4,Коммерческие,0,0.062696
2,00035b28bedc6d3a0336cb61a841d79b,7.214286,14,0.000000,0.000000,Науки о данных,2,По межправительственным соглашениям,0,0.006225
3,00044a704df898b9a87faf40795d62b6,8.454545,22,0.000000,0.000000,Государственно-конфессиональные отношения. Пра...,2,Бюджетные,0,0.121903
4,0004d03d5c41e468a521ed8a964fe889,7.250000,16,0.062500,0.000000,Управление развитием бизнеса,2,Бюджетные,0,-0.377627
...,...,...,...,...,...,...,...,...,...,...
62925,fffaf96349866c4fbf94dca72aad6f0e,6.709091,55,0.018182,0.000000,Международный бакалавриат по бизнесу и экономике,4,Бюджетные,0,0.002415
62926,fffbf91749bd6fd12d3ddf2ee81ee633,8.238095,21,0.000000,0.000000,Юриспруденция,2,Бюджетные,0,-0.244714
62927,fffc29ec23eda92a88df30e449afcf1c,6.520000,25,0.000000,0.000000,Экономика,4,Коммерческие,0,-0.076814
62928,fffe12fc2c520bcb03f1437defdc7ecf,7.600000,25,0.000000,0.000000,Маркетинг,2,Бюджетные,0,-0.328200


In [12]:


categorical_cols = ['course', 'program', 'place_type']


encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_array = encoder.fit_transform(df_encoded[categorical_cols])

encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(categorical_cols)
)



In [13]:
df_final = pd.concat([df_encoded.drop(columns=categorical_cols), encoded_df], axis=1)
df_final

,student__id_hash,avg_grade,count_grades,proportion_retake,proportion_retake_com,student_status,pearson,course_1,course_2,course_3,...,place_type_Внеконкурсное поступление,place_type_Коммерческие,place_type_Коммерческие за счет средств вуза,place_type_Коммерческие за счет средств вуза для иностранных граждан,place_type_Коммерческие места для иностранных граждан,place_type_Места с оплатой стоимости обучения на договорной основе,"place_type_Места, обеспеченные государственным финансированием",place_type_По межправительственным соглашениям,place_type_Сетевой студент,place_type_Целевые
0,00000d86dad0d3c197341f776bb61a1d,8.166667,24,0.000000,0.000000,0,-0.104406,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0001dd443aff5c0800694730323c3d7d,6.973913,115,0.034783,0.008696,0,0.062696,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,00035b28bedc6d3a0336cb61a841d79b,7.214286,14,0.000000,0.000000,0,0.006225,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,00044a704df898b9a87faf40795d62b6,8.454545,22,0.000000,0.000000,0,0.121903,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0004d03d5c41e468a521ed8a964fe889,7.250000,16,0.062500,0.000000,0,-0.377627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62925,fffaf96349866c4fbf94dca72aad6f0e,6.709091,55,0.018182,0.000000,0,0.002415,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
62926,fffbf91749bd6fd12d3ddf2ee81ee633,8.238095,21,0.000000,0.000000,0,-0.244714,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
62927,fffc29ec23eda92a88df30e449afcf1c,6.520000,25,0.000000,0.000000,0,-0.076814,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
62928,fffe12fc2c520bcb03f1437defdc7ecf,7.600000,25,0.000000,0.000000,0,-0.328200,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# df_encoded_2 = df_encoded[df_encoded['count_grades'] > 100]

сделать тренд оценок (разница оценок)   
количество несдавших эту же дисциплину в его группе

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score

In [21]:
X = df_final.drop(columns=['student__id_hash', 'student_status'])
y = df_final['student_status']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)


model = LogisticRegression(max_iter=100000000000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print (accuracy_score(y_test, y_pred))

print(precision_score(y_test, y_pred))

0.9896710630859685
0.900398406374502


In [24]:
y_pred_const = np.zeros(len(X_test))
print (accuracy_score(y_test, y_pred_const))

print(precision_score(y_test, y_pred_const))


0.9699136606811801
0.0


/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [16]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,           # глубина дерева (ограничиваем, чтобы избежать переобучения)
    min_samples_split=10,  # мин. объектов для разделения узла
    min_samples_leaf=10,    # мин. объектов в листе
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print (accuracy_score(y_test, y_pred))

print(precision_score(y_test, y_pred))

0.9888235605699455
0.8846960167714885


In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=50,      # количество деревьев
    max_depth=20,          # максимальная глубина
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print (accuracy_score(y_test, y_pred))

print(precision_score(y_test, y_pred))

0.989618094178717
0.9244444444444444


In [20]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print (accuracy_score(y_test, y_pred))

print(precision_score(y_test, y_pred))

0.9898299698077229
0.8856589147286822


In [55]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 10.1 MB/s  0:00:13m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 10.6 MB/s  0:00:28m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [18]:
# pip install xgboost
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


print (accuracy_score(y_test, y_pred))

print(precision_score(y_test, y_pred))

0.9903066899729859
0.8864970645792564


In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Ваши данные
X = df_final.drop(columns=['student__id_hash', 'student_status'])
y = df_final['student_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Словарь моделей
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB()
}

# Сравнение моделей
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Для ROC-AUC нужно probability=True для некоторых моделей
    if hasattr(model, "predict_proba"):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_pred_proba)
    else:
        roc_auc = np.nan
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred, average='weighted'),
        'ROC-AUC': roc_auc
    })

# Результаты
results_df = pd.DataFrame(results)
print(results_df.round(4))

# Находим лучшую модель
best_model = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
print(f"\nЛучшая модель по accuracy: {best_model}")

                 Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
0  Logistic Regression    0.9893     0.9888  0.9893    0.9888   0.9829
1        Decision Tree    0.9891     0.9887  0.9891    0.9889   0.9419
2        Random Forest    0.9782     0.9784  0.9782    0.9728   0.9804
3    Gradient Boosting    0.9896     0.9891  0.9896    0.9892   0.9820
4                  SVM    0.9836     0.9824  0.9836    0.9817   0.9533
5                  KNN    0.9873     0.9866  0.9873    0.9866   0.9240
6          Naive Bayes    0.1298     0.9587  0.1298    0.1820   0.5352

Лучшая модель по accuracy: Gradient Boosting


In [58]:
df_encoded

,student__id_hash,avg_grade,count_grades,proportion_retake,proportion_retake_com,program,course,place_type,student_status
0,00000d86dad0d3c197341f776bb61a1d,8.166667,24,0.000000,0.000000,Математика,2,Бюджетные,0
1,0001dd443aff5c0800694730323c3d7d,6.973913,115,0.034783,0.008696,Реклама и связи с общественностью,4,Коммерческие,0
2,00035b28bedc6d3a0336cb61a841d79b,7.214286,14,0.000000,0.000000,Науки о данных,2,По межправительственным соглашениям,0
3,00044a704df898b9a87faf40795d62b6,8.454545,22,0.000000,0.000000,Государственно-конфессиональные отношения. Пра...,2,Бюджетные,0
4,0004d03d5c41e468a521ed8a964fe889,7.250000,16,0.062500,0.000000,Управление развитием бизнеса,2,Бюджетные,0
...,...,...,...,...,...,...,...,...,...
62925,fffaf96349866c4fbf94dca72aad6f0e,6.709091,55,0.018182,0.000000,Международный бакалавриат по бизнесу и экономике,4,Бюджетные,0
62926,fffbf91749bd6fd12d3ddf2ee81ee633,8.238095,21,0.000000,0.000000,Юриспруденция,2,Бюджетные,0
62927,fffc29ec23eda92a88df30e449afcf1c,6.520000,25,0.000000,0.000000,Экономика,4,Коммерческие,0
62928,fffe12fc2c520bcb03f1437defdc7ecf,7.600000,25,0.000000,0.000000,Маркетинг,2,Бюджетные,0


In [68]:
counts = df_encoded.groupby(['course', 'program']).size().reset_index(name='student_count')

# print(counts)
max_programs = counts.loc[counts['student_count'].idxmax()]

print(max_programs)


course                4
program          Дизайн
student_count      1975
Name: 605, dtype: object


In [1]:
for course in counts['course'].unique():
    top = counts[counts['course'] == course].nlargest(1, 'student_count')
    print(f"Курс {course}: {top['program'].values[0]} ({top['student_count'].values[0]} студентов)")

NameError: name 'counts' is not defined

In [5]:
def make_data_from_program(df : pd.DataFrame, program: str, addictional_columns: list) -> None:
    df_baseline = df[df['program'] == program]
    
    
    
    df_baseline_encoded = df_baseline.pivot_table(
        index='student_id_hash',
        columns=['course', 'module', 'subject_name'],
        values='grade_10',
        aggfunc = 'mean'
    )

    df_baseline_encoded.columns = [
        "_".join(map(str, col)).strip()
        for col in df_baseline_encoded.columns
    ]
    df_baseline_encoded = df_baseline_encoded.reset_index()

    df_baseline_encoded = df_baseline_encoded.merge(
        df_baseline[['student_id_hash'] + addictional_columns],
        on='student_id_hash',
        how='left'
    )
    df_baseline_encoded = df_baseline_encoded.drop_duplicates()
    return df_baseline_encoded


In [ ]:
# df = pd.read_csv('../data/raw/grades.csv', sep= ';')


/tmp/ipykernel_86428/2259448426.py:1: DtypeWarning: Columns (0: course, 1: module, 2: academic_year) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/grades.csv', sep= ';')


In [9]:
df_design = make_data_from_program(df_clean, 'Дизайн', ['student_status'])

: 